In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from nba_api.stats.endpoints import scoreboardv2
import nba_api_module as nbam
import time
import json

team_id_dict = nbam.TEAM_IDS
team_abr_dict = nbam.TEAM_ABBREVIATIONS

def get_upcoming_games(days_ahead=3):
    """
    Lekéri a következő N napra tervezett meccseket.
    Maximum 1 meccs csapatonként.
    """
    upcoming = []
    teams_seen = set()
    
    today = datetime.now()
    
    for day_offset in range(days_ahead):
        check_date = today + timedelta(days=day_offset)
        date_str = check_date.strftime('%Y-%m-%d')
        
        print(f"\nEllenőrzés: {date_str}")
        
        try:
            scoreboard = scoreboardv2.ScoreboardV2(game_date=date_str)
            games = scoreboard.get_data_frames()[0]  # GameHeader
            
            if len(games) == 0:
                print(f"  Nincs meccs ezen a napon")
                continue
            
            for _, game in games.iterrows():
                game_id = str(game['GAME_ID'])
                home_team_id = game['HOME_TEAM_ID']
                home_team = team_id_dict[home_team_id]
                home_team_abr = team_abr_dict[home_team]
                away_team_id = game['VISITOR_TEAM_ID']
                away_team = team_id_dict[away_team_id]
                away_team_abr = team_abr_dict[away_team]
                
                # Csak akkor adjuk hozzá, ha egyik csapat sem szerepelt még
                if home_team_id not in teams_seen and away_team_id not in teams_seen:
                    upcoming.append({
                        'game_id': game_id,
                        'game_date': date_str,
                        'home_team_id': home_team_id,
                        'away_team_id': away_team_id,
                        'home_team': home_team,
                        'away_team': away_team
                    })
                    teams_seen.add(home_team_id)
                    teams_seen.add(away_team_id)
                    print(f"  ✓ {home_team_abr} vs. {away_team_abr} (ID: {game_id})")
            
            time.sleep(1)  # Rate limit
            
        except Exception as e:
            print(f"  Hiba {date_str} lekérésekor: {e}")
            continue
    
    print(f"\n{'='*50}")
    print(f"Összesen {len(upcoming)} meccs találva")
    print(f"{'='*50}")
    
    return pd.DataFrame(upcoming)


def create_pregame_features(game_id, game_date):
    """
    Egy adott meccsre létrehozza az összes pregame feature-t.
    """
    print(f"\n[{game_id}] Pregame features létrehozása...")
    
    # Extract meta info
    meta = nbam.extract_game_meta(game_id)
    
    # Override date with actual game date (mivel még nem játszották le)
    meta['date'] = game_date
    
    features = {}
    
    # 1) Pregame stats
    try:
        pregame = nbam.extract_pregame(game_id)
        features.update(pregame)
        print(f"  ✓ Pregame stats")
    except Exception as e:
        print(f"  ✗ Pregame stats hiba: {e}")
        return None
    
    time.sleep(1)
    
    # 2) Injury data
    try:
        injury = nbam.extract_injury(game_id)
        # Eltávolítjuk a missing_starters feature-t (mint az ml.ipynb-ben)
        injury.pop('home_missing_starters', None)
        injury.pop('away_missing_starters', None)
        features.update(injury)
        print(f"  ✓ Injury data")
    except Exception as e:
        print(f"  ✗ Injury data hiba: {e}")
        # Ha nincs injury adat, nullázzuk
        features['home_injury_count'] = 0
        features['away_injury_count'] = 0
    
    time.sleep(1)
    
    # 3) Advanced stats
    try:
        advanced = nbam.extract_advanced_stats(game_id)
        features.update(advanced)
        print(f"  ✓ Advanced stats")
    except Exception as e:
        print(f"  ✗ Advanced stats hiba: {e}")
        return None
    
    time.sleep(1)
    
    # 4) Form metrics
    try:
        form = nbam.extract_form(game_id)
        features.update(form)
        print(f"  ✓ Form metrics")
    except Exception as e:
        print(f"  ✗ Form metrics hiba: {e}")
        return None
    
    return features


def calculate_differential_features(features_dict):
    """
    Differenciális feature-ök hozzáadása (mint az ml.ipynb feature engineering cellájában)
    """
    df = pd.DataFrame([features_dict])
    
    # Offensive/Defensive Rating különbségek
    df['ORtg_diff'] = df['home_ORtg'] - df['away_ORtg']
    df['DRtg_diff'] = df['home_DRtg'] - df['away_DRtg']
    df['NET_rtg_diff'] = df['home_NET_rtg'] - df['away_NET_rtg']
    
    # Pace különbség
    df['PACE_diff'] = df['home_PACE'] - df['away_PACE']
    
    # Hatékonyság különbségek
    df['TS_diff'] = df['home_TS%'] - df['away_TS%']
    df['EFG_diff'] = df['home_EFG%'] - df['away_EFG%']
    df['AST_ratio_diff'] = df['home_AST_ratio'] - df['away_AST_ratio']
    df['OREB_diff'] = df['home_OREB%'] - df['away_OREB%']
    df['turnover_diff'] = df['home_turnover_ratio'] - df['away_turnover_ratio']
    
    # Játékos minőség különbségek
    df['starter_PER_diff'] = df['home_starter_avg_PER'] - df['away_starter_avg_PER']
    df['bench_PER_diff'] = df['home_bench_avg_PER'] - df['away_bench_avg_PER']
    df['star_usage_diff'] = df['home_star_usage'] - df['away_star_usage']
    df['avg_TS_diff'] = df['home_avg_TS'] - df['away_avg_TS']
    df['top3_points_diff'] = df['home_top3_points_avg'] - df['away_top3_points_avg']
    
    # Pihenés és forma különbségek
    df['rest_days_diff'] = df['home_rest_days'] - df['away_rest_days']
    df['recent_form10_diff'] = df['home_recent_form10'] - df['away_recent_form10']
    df['recent_form5_diff'] = df['home_recent_form5'] - df['away_recent_form5']
    df['recent_form3_diff'] = df['home_recent_form3'] - df['away_recent_form3']
    
    # Sérülés különbség
    df['injury_count_diff'] = df['home_injury_count'] - df['away_injury_count']
    
    # Back-to-back advantage
    df['b2b_advantage'] = df['away_is_back_to_back'].astype(int) - df['home_is_back_to_back'].astype(int)
    
    return df.iloc[0].to_dict()


def align_features_to_model(features_dict, feature_columns_path='feature_columns.json'):
    """
    Biztosítja, hogy a feature-ök pontosan egyezzenek a modell által elvárt feature listával.
    """
    with open(feature_columns_path, 'r') as f:
        expected_features = json.load(f)
    
    # Ellenőrizzük, hogy minden szükséges feature megvan-e
    missing_features = [f for f in expected_features if f not in features_dict]
    if missing_features:
        print(f"\n⚠️  Hiányzó features: {missing_features}")
        # Nullával töltjük fel a hiányzó feature-öket
        for feat in missing_features:
            features_dict[feat] = 0
    
    # Csak a modell által elvárt feature-öket tartjuk meg, megfelelő sorrendben
    aligned_features = {feat: features_dict[feat] for feat in expected_features}
    
    return pd.DataFrame([aligned_features])


def prepare_upcoming_games_for_prediction(days_ahead=3, output_file='data/upcoming_games_features.csv'):
    """
    Teljes pipeline: lekéri a következő meccseket és előkészíti a predikcióhoz.
    """
    print("="*60)
    print("KÖVETKEZŐ MECCSEK ELŐKÉSZÍTÉSE PREDIKCIÓHOZ")
    print("="*60)
    
    # 1) Következő meccsek lekérése
    upcoming_df = get_upcoming_games(days_ahead=days_ahead)
    
    if len(upcoming_df) == 0:
        print("\n❌ Nem találtunk következő meccseket!")
        return None
    
    # 2) Features gyűjtése minden meccshez
    all_features = []
    
    for idx, game in upcoming_df.iterrows():
        game_id = game['game_id']
        game_date = game['game_date']
        
        print(f"\n{'='*60}")
        print(f"[{idx+1}/{len(upcoming_df)}] {game['away_team']} @ {game['home_team']}")
        print(f"{'='*60}")
        
        try:
            # Pregame features
            features = create_pregame_features(game_id, game_date)
            
            if features is None:
                print(f"  ⚠️  Kihagyva (hiányos adatok)")
                continue
            
            # Differenciális features
            features = calculate_differential_features(features)
            
            # Game meta info hozzáadása
            features['game_id'] = game_id
            features['game_date'] = game_date
            features['home_team'] = game['home_team']
            features['away_team'] = game['away_team']
            
            all_features.append(features)
            
            print(f"  ✅ Sikeres feldolgozás")
            
        except Exception as e:
            print(f"  ❌ Hiba: {e}")
            continue
        
        time.sleep(2)  # Rate limit
    
    if len(all_features) == 0:
        print("\n❌ Egyetlen meccshez sem sikerült adatot gyűjteni!")
        return None
    
    # 3) DataFrame összeállítása
    features_df = pd.DataFrame(all_features)
    
    # 4) Feature alignment a modell által elvárt formátumra
    print(f"\n{'='*60}")
    print("FEATURE ALIGNMENT")
    print(f"{'='*60}")
    
    # Meta információkat külön tároljuk
    meta_cols = ['game_id', 'game_date', 'home_team', 'away_team']
    meta_df = features_df[meta_cols].copy()
    
    # Feature-ök alignment
    aligned_features = []
    for idx, row in features_df.iterrows():
        row_dict = row.to_dict()
        aligned = align_features_to_model(row_dict)
        aligned_features.append(aligned)
    
    aligned_df = pd.concat(aligned_features, ignore_index=True)
    
    # Meta info visszacsatolása
    final_df = pd.concat([meta_df.reset_index(drop=True), aligned_df], axis=1)
    
    # 5) Mentés
    final_df.to_csv(output_file, index=False)
    
    print(f"\n{'='*60}")
    print("✅ KÉSZ!")
    print(f"{'='*60}")
    print(f"Meccsek száma: {len(final_df)}")
    print(f"Feature-ök száma: {len(aligned_df.columns)}")
    print(f"Mentve: {output_file}")
    print(f"\nMeccsek:")
    for _, game in final_df.iterrows():
        print(f"  • {game['away_team']} @ {game['home_team']} ({game['game_date']})")
    
    return final_df


# HASZNÁLAT:
upcoming_features = prepare_upcoming_games_for_prediction(days_ahead=3)

upcoming_features

KÖVETKEZŐ MECCSEK ELŐKÉSZÍTÉSE PREDIKCIÓHOZ

Ellenőrzés: 2025-11-21
  ✓ CLE vs. IND (ID: 0022500048)
  ✓ BOS vs. BKN (ID: 0022500049)
  ✓ TOR vs. WAS (ID: 0022500050)
  ✓ CHI vs. MIA (ID: 0022500051)
  ✓ DAL vs. NOP (ID: 0022500052)
  ✓ PHX vs. MIN (ID: 0022500053)
  ✓ HOU vs. DEN (ID: 0022500054)
  ✓ UTA vs. OKC (ID: 0022500055)
  ✓ GSW vs. POR (ID: 0022500056)

Ellenőrzés: 2025-11-22
  ✓ CHA vs. LAC (ID: 0022500268)
  ✓ ORL vs. NYK (ID: 0022500269)
  ✓ MIL vs. DET (ID: 0022500272)

Ellenőrzés: 2025-11-23

Összesen 12 meccs találva

[1/12] Indiana Pacers @ Cleveland Cavaliers

[0022500048] Pregame features létrehozása...
  ❌ Hiba: 'NoneType' object has no attribute 'get'

[2/12] Brooklyn Nets @ Boston Celtics

[0022500049] Pregame features létrehozása...
  ❌ Hiba: 'NoneType' object has no attribute 'get'

[3/12] Washington Wizards @ Toronto Raptors

[0022500050] Pregame features létrehozása...
  ❌ Hiba: 'NoneType' object has no attribute 'get'

[4/12] Miami Heat @ Chicago Bulls

[0022

In [1]:
from nba_api_module import TEAM_IDS 

TEAM_IDS

{1610612737: 'Atlanta Hawks',
 1610612738: 'Boston Celtics',
 1610612739: 'Cleveland Cavaliers',
 1610612740: 'New Orleans Pelicans',
 1610612741: 'Chicago Bulls',
 1610612742: 'Dallas Mavericks',
 1610612743: 'Denver Nuggets',
 1610612744: 'Golden State Warriors',
 1610612745: 'Houston Rockets',
 1610612746: 'LA Clippers',
 1610612747: 'Los Angeles Lakers',
 1610612748: 'Miami Heat',
 1610612749: 'Milwaukee Bucks',
 1610612750: 'Minnesota Timberwolves',
 1610612751: 'Brooklyn Nets',
 1610612752: 'New York Knicks',
 1610612753: 'Orlando Magic',
 1610612754: 'Indiana Pacers',
 1610612755: 'Philadelphia 76ers',
 1610612756: 'Phoenix Suns',
 1610612757: 'Portland Trail Blazers',
 1610612758: 'Sacramento Kings',
 1610612759: 'San Antonio Spurs',
 1610612760: 'Oklahoma City Thunder',
 1610612761: 'Toronto Raptors',
 1610612762: 'Utah Jazz',
 1610612763: 'Memphis Grizzlies',
 1610612764: 'Washington Wizards',
 1610612765: 'Detroit Pistons',
 1610612766: 'Charlotte Hornets'}